# Notebook 05 — Final Load Preparation for Tableau

**Objective:** Produce the final, Tableau-optimised output files from the cleaned dataset. All exports land in `data/processed/` so the submission checklist and local notebook runs stay in sync.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(
    os.environ.get(
        'PROJECT_ROOT',
        Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
    )
).resolve()

def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]

RAW_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv',
    PROJECT_ROOT / 'investments_VC.csv',
    Path('/content/investments_VC.csv'),
)

CLEAN_DATA_PATH = first_existing(
    PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv',
    PROJECT_ROOT / 'startups_cleaned.csv',
    Path('/content/startups_cleaned.csv'),
)

SCREENSHOT_DIR = PROJECT_ROOT / 'tableau' / 'screenshots'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    _colab_files = None
    IN_COLAB = False

def maybe_download(path: Path) -> None:
    if IN_COLAB:
        _colab_files.download(str(path))


df = pd.read_csv(CLEAN_DATA_PATH, low_memory=False)
df = df.rename(columns={c: c.lower() if c.startswith('round_') else c for c in df.columns})

print(f'Loaded cleaned data: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Outputs will be saved to: {OUTPUT_DIR}')


In [ ]:
tableau_cols = [
    'name', 'status', 'country_code', 'market', 'primary_category',
    'founding_decade', 'founded_year', 'founded_at', 'first_funding_at', 'last_funding_at',
    'funding_rounds', 'funding_total_usd', 'avg_funding_per_round',
    'seed', 'venture', 'angel', 'round_a', 'round_b', 'round_c',
    'days_to_first_funding', 'funding_duration_days',
    'is_closed', 'is_usa', 'reached_series_a', 'funding_tier', 'has_seed'
]

df_tableau = df[[c for c in tableau_cols if c in df.columns]].copy()
rename_map = {
    'name': 'Startup Name',
    'status': 'Status',
    'country_code': 'Country',
    'market': 'Market',
    'primary_category': 'Primary Category',
    'founding_decade': 'Founding Decade',
    'founded_year': 'Founded Year',
    'founded_at': 'Founded Date',
    'first_funding_at': 'First Funding Date',
    'last_funding_at': 'Last Funding Date',
    'funding_rounds': 'Funding Rounds',
    'funding_total_usd': 'Total Funding USD',
    'avg_funding_per_round': 'Avg Funding Per Round USD',
    'seed': 'Seed Amount USD',
    'venture': 'Venture Amount USD',
    'angel': 'Angel Amount USD',
    'round_a': 'Series A USD',
    'round_b': 'Series B USD',
    'round_c': 'Series C USD',
    'days_to_first_funding': 'Days to First Funding',
    'funding_duration_days': 'Funding Duration Days',
    'is_closed': 'Closed (0/1)',
    'is_usa': 'USA Based (0/1)',
    'reached_series_a': 'Reached Series A (0/1)',
    'funding_tier': 'Funding Tier',
    'has_seed': 'Has Seed (0/1)',
}
df_tableau.rename(columns=rename_map, inplace=True)

out_path = OUTPUT_DIR / 'tableau_master.csv'
df_tableau.to_csv(out_path, index=False)

print(f'Master Tableau dataset exported: {out_path}')
print(f'Shape: {df_tableau.shape[0]:,} rows x {df_tableau.shape[1]} columns')
df_tableau.head(3)

In [ ]:
total = len(df)

kpi_data = {
    'KPI': [
        'Total Startups Analysed',
        'Overall Failure Rate (%)',
        'Operating Rate (%)',
        'Acquisition Rate (%)',
        'IPO Rate (%)',
        'Median Funding — Closed ($)',
        'Median Funding — Operating ($)',
        'Funding Gap (Operating vs Closed) ($)',
        'Avg Funding Rounds — Closed',
        'Avg Funding Rounds — Operating',
        'Avg Funding Rounds — Acquired',
        'Series A Failure Rate (%)',
        'Pre-Series A Failure Rate (%)',
        'USA Startup Failure Rate (%)',
        'Non-USA Startup Failure Rate (%)',
    ],
    'Value': [
        total,
        round((df['status'] == 'closed').sum() / total * 100, 2),
        round((df['status'] == 'operating').sum() / total * 100, 2),
        round((df['status'] == 'acquired').sum() / total * 100, 2),
        round((df['status'] == 'ipo').sum() / total * 100, 2),
        int(df[df['status'] == 'closed']['funding_total_usd'].median()),
        int(df[df['status'] == 'operating']['funding_total_usd'].median()),
        int(df[df['status'] == 'operating']['funding_total_usd'].median() - df[df['status'] == 'closed']['funding_total_usd'].median()),
        round(df[df['status'] == 'closed']['funding_rounds'].mean(), 2),
        round(df[df['status'] == 'operating']['funding_rounds'].mean(), 2),
        round(df[df['status'] == 'acquired']['funding_rounds'].mean(), 2),
        round(df[df['reached_series_a'] == 1]['is_closed'].mean() * 100, 2),
        round(df[df['reached_series_a'] == 0]['is_closed'].mean() * 100, 2),
        round(df[df['country_code'] == 'USA']['is_closed'].mean() * 100, 2),
        round(df[df['country_code'] != 'USA']['is_closed'].mean() * 100, 2),
    ],
}

df_kpi = pd.DataFrame(kpi_data)
kpi_path = OUTPUT_DIR / 'kpi_summary.csv'
df_kpi.to_csv(kpi_path, index=False)

print(f'KPI summary exported: {kpi_path}')
print(df_kpi.to_string(index=False))

In [ ]:
country_agg = df.groupby('country_code').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Median_Funding_USD=('funding_total_usd', 'median'),
    Avg_Rounds=('funding_rounds', 'mean'),
).reset_index()
country_agg['Failure_Rate_Pct'] = (country_agg['Closed'] / country_agg['Total_Startups'] * 100).round(2)
country_agg = country_agg[country_agg['Total_Startups'] >= 30].sort_values('Failure_Rate_Pct', ascending=False)

country_path = OUTPUT_DIR / 'country_level_summary.csv'
country_agg.to_csv(country_path, index=False)

print(f'Country summary exported: {country_path}')
print(country_agg.head(15).to_string(index=False))

In [ ]:
sector_agg = df.groupby('market').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Avg_Rounds=('funding_rounds', 'mean'),
    Series_A_Rate=('reached_series_a', 'mean'),
).reset_index()
sector_agg['Failure_Rate_Pct'] = (sector_agg['Closed'] / sector_agg['Total_Startups'] * 100).round(2)
sector_agg['Series_A_Rate_Pct'] = (sector_agg['Series_A_Rate'] * 100).round(2)
sector_agg = sector_agg[sector_agg['Total_Startups'] >= 50].sort_values('Failure_Rate_Pct', ascending=False)

sector_path = OUTPUT_DIR / 'sector_level_summary.csv'
sector_agg.to_csv(sector_path, index=False)

print(f'Sector summary exported: {sector_path}')
print(sector_agg.head(20).to_string(index=False))

In [ ]:
yearly_agg = df[df['founded_year'].between(1990, 2014)].groupby('founded_year').agg(
    Total_Startups=('is_closed', 'count'),
    Closed=('is_closed', 'sum'),
    Avg_Funding_USD=('funding_total_usd', 'mean'),
    Avg_Rounds=('funding_rounds', 'mean'),
).reset_index()
yearly_agg['Failure_Rate_Pct'] = (yearly_agg['Closed'] / yearly_agg['Total_Startups'] * 100).round(2)

yearly_path = OUTPUT_DIR / 'yearly_trend_summary.csv'
yearly_agg.to_csv(yearly_path, index=False)

print(f'Yearly trend exported: {yearly_path}')
print(yearly_agg.to_string(index=False))

In [ ]:
outputs = [
    ('tableau_master.csv', 'Main dashboard feed — all startup records'),
    ('kpi_summary.csv', 'Executive KPI table for dashboard header'),
    ('country_level_summary.csv', 'Geographic map view'),
    ('sector_level_summary.csv', 'Sector drill-down view'),
    ('yearly_trend_summary.csv', 'Time trend line chart'),
]

print('=== FINAL LOAD SUMMARY ===')
for fname, desc in outputs:
    path = OUTPUT_DIR / fname
    if path.exists():
        rows = sum(1 for _ in open(path, 'r', encoding='utf-8')) - 1
        size_kb = path.stat().st_size / 1024
        print(f'  ✓ {fname:<30} {rows:>6,} rows | {size_kb:>7.1f} KB | {desc}')
    else:
        print(f'  ✗ {fname} — NOT FOUND')

print('\nAll files are ready for Tableau import.')
print('Connect tableau_master.csv as the primary data source.')
print('Use the summary files for KPI tiles and pre-aggregated views.')